# 미션 8 — U-Net 기반 축구 경기 Semantic Segmentation (11 클래스)

축구 경기 장면 이미지의 **모든 픽셀**을 11개 클래스(골대, 심판, 광고판, 그라운드, 공, 코치·관계자, 관중, 골키퍼 A/B, 팀 A/B) 중 하나로 분류하는 U-Net 모델을 직접 설계·학습·평가하고, 증강·하이퍼파라미터·구조 변경으로 성능을 개선합니다.

**데이터**: [Kaggle — Football Semantic Segmentation](https://www.kaggle.com/datasets/sadhliroomyprime/football-semantic-segmentation) (경기 프레임 약 100장 + 색상 코드 마스크)

## 노트북 구성

| 단계 | 내용 |
|---|---|
| 0 | 환경 설정 · 전역 하이퍼파라미터 |
| 1 | 데이터 다운로드 · 이미지/마스크 짝 맞추기 |
| 2 | EDA — 색상 팔레트 → 클래스 매핑, 클래스 분포 |
| 3 | 전처리 — RGB 마스크 → 클래스 인덱스, 리사이즈, Train/Val/Test 분할 |
| 4 | Dataset · 데이터 증강 |
| 5 | 모델 — U-Net / Attention U-Net (직접 구현) |
| 6 | 손실 함수(CE, Dice) · 평가 지표(Pixel Acc, IoU, Dice) |
| 7 | 학습 · 평가 루프 |
| 8 | 실험 E1~E6 (Baseline → 손실 → 증강 → 튜닝 → 구조 개선) |
| 9 | 최종 모델 선정 · 테스트 평가 · 시각화 · 저장 |
| 10 | 결과 공유용 요약 출력 |

## 실험 설계 개요와 전체 가설

한 번에 **한 가지 요인만** 바꿔 효과를 분리해 측정합니다.

| 실험 | 모델 | 손실 | 증강 | 바꾸는 요인 | 가설 |
|---|---|---|---|---|---|
| E1 | U-Net | CE | ✗ | (기준선) | 대면적 클래스(그라운드·관중·광고판)는 잘 맞히지만 공·골대·심판은 거의 못 맞힐 것 |
| E2 | U-Net | CE + Dice | ✗ | 손실 함수 | Dice가 클래스별 겹침을 직접 최적화해 소수 클래스 IoU↑ → mIoU↑ (Pixel Acc는 비슷) |
| E3 | U-Net | CE + Dice | ✓ | 데이터 증강 | 학습 데이터가 70장뿐이라 과적합이 심함 → 증강으로 Val/Test 격차 축소, mIoU↑ |
| E4 | U-Net | CE + Dice | ✓ | lr · batch | 소량 데이터에선 작은 배치·적절한 lr이 일반화에 유리할 것 |
| E5 | Attention U-Net | CE + Dice | ✓ | 구조(Attention Gate) | skip 연결의 배경 노이즈 억제 → 작은 객체(공·심판) 소폭 개선, 전체 차이는 작을 수 있음 |
| E6 | U-Net++ (ResNet18, ImageNet 사전학습) | CE + Dice | ✓ | 구조 + 사전학습 | 소량 데이터에선 **사전학습 인코더 효과**가 가장 커서 최고 성능을 낼 것 |

### 전체 관전 포인트
1. **mIoU vs Pixel Accuracy의 괴리** — 그라운드가 픽셀 대부분을 차지하므로 Pixel Acc는 처음부터 높게 나옵니다. 진짜 실력은 mIoU와 **클래스별 IoU**에서 드러납니다.
2. **혼동 쌍** — Team A ↔ Goalkeeper A, Team A ↔ Team B(유니폼 색), Referee ↔ Coaches & Officials, Goal Bar ↔ Advertisement/Audience. 혼동행렬에서 어디가 섞이는지 봅니다.
3. **학습 곡선의 Train/Val 격차** — 데이터가 적어 과적합 시점(best epoch)이 언제인지, 증강 후 어떻게 바뀌는지.
4. **테스트 세트가 약 15장**뿐이므로 실험 간 1~2%p 차이는 우연일 수 있습니다. 최종 모델 선택은 Test가 아닌 **Validation mIoU**로 하고, Test는 최종 보고에만 씁니다(정보 누수 방지).

## 실행 환경: Windows 11 · VS Code · CPU

CPU 학습은 GPU보다 수십 배 느리므로, 미션 요구사항은 모두 유지하면서 **CPU에 맞게 규모를 줄였습니다**.

| 항목 | GPU 기준 일반 설정 | **본 노트북(CPU)** | 이유 |
|---|---|---|---|
| 입력 해상도 | 512×288 | **384×224** | 연산량 약 40% 감소 (32의 배수라 U-Net++도 가능) |
| U-Net 기본 채널 | 32 | **16** | 연산량·파라미터 약 1/4, 소량 데이터 과적합도 완화 |
| 최대 에폭 / 조기 종료 | 80 / 20 | **40 / 10** | 실험당 학습 시간 제한 |
| 튜닝용 에폭 | 40 | **12** | 4개 조합 탐색 |
| E6 인코더 | ResNet34 | **ResNet18** + 가벼운 디코더 | U-Net++ 디코더는 특히 무거움 |
| DataLoader workers | 2 | **0** | Windows + Jupyter에서 멀티프로세스 로더가 멈추는 문제 방지 |

- **예상 시간**: CPU 성능에 따라 실험 1개당 수십 분, 전체 2~5시간 수준입니다. **섹션 7의 속도 측정 셀**이 실제 PC 기준 예상 시간을 계산해 주므로, 그 결과를 보고 `RUN_TUNING`, `RUN_E5`, `RUN_E6` 플래그로 실험 범위를 조절하세요.
- 흐름만 먼저 확인하려면 `QUICK_MODE = True` (에폭 5)로 한 번 돌려 오류가 없는지 본 뒤 본 실행을 권장합니다.
- 각 실험이 끝날 때마다 결과가 `outputs/`에 CSV로 저장되므로, 중간에 커널이 멈춰도 그때까지의 결과는 남습니다.
- 해상도를 낮춘 만큼 **공(Ball)은 지름 2~4픽셀** 수준까지 작아집니다 → 공의 IoU는 매우 낮게 나올 것으로 예상합니다(CPU 설정의 한계로 해석).

### VS Code 준비 (최초 1회)
1. Python 3.10~3.12 설치, VS Code에 **Python**·**Jupyter** 확장 설치
2. 터미널에서 가상환경 생성: `python -m venv .venv` → 노트북 오른쪽 위 **커널 선택**에서 `.venv` 지정
3. 아래 0-1 셀이 필요한 패키지를 설치합니다(Windows용 기본 `torch`는 CPU 버전).

## 0. 환경 설정

- `kagglehub`: Kaggle 데이터셋을 인증 없이(공개 데이터셋) 내려받는 공식 라이브러리
- `segmentation-models-pytorch`(smp): E6의 U-Net++ + 사전학습 인코더에만 사용. **E1~E5의 U-Net/Attention U-Net은 직접 구현**합니다.
- 재현성을 위해 모든 난수 시드를 고정합니다. GPU가 있으면 자동으로 AMP(혼합 정밀도)를 쓰고, CPU에서는 모든 코어를 사용하도록 스레드 수를 지정합니다.
- 그래프의 한글이 깨지지 않도록 Windows 기본 한글 폰트(맑은 고딕)를 지정합니다.

In [ ]:
# ===== 0-1. 패키지 설치 (현재 선택된 커널의 가상환경에 설치) =====
# 이미 설치되어 있으면 빠르게 넘어갑니다. 설치 후에는 커널을 한 번 재시작하세요.
%pip install -q torch torchvision opencv-python numpy pandas matplotlib scipy tqdm ipywidgets kagglehub segmentation-models-pytorch

In [ ]:
# ===== 0-2. 라이브러리 임포트 및 전역 설정 =====
import os, glob, json, random, time, re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# ---------------- 재현성: 시드 고정 ----------------
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # 속도 우선 (완전 재현이 필요하면 True)
    torch.backends.cudnn.benchmark = True
seed_everything()

# ---------------- 디바이스 ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"            # GPU일 때만 혼합 정밀도 사용
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count())  # CPU 전체 코어 사용
print("torch:", torch.__version__, "| device:", DEVICE, "| CPU threads:", torch.get_num_threads(),
      "| GPU:", torch.cuda.get_device_name(0) if USE_AMP else "-")

# ---------------- 한글 폰트 (Windows: 맑은 고딕) ----------------
import platform
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# ---------------- 전역 하이퍼파라미터 ----------------
QUICK_MODE   = False          # True면 에폭을 크게 줄여 오류 없이 끝까지 도는지만 빠르게 확인
IMG_H, IMG_W = 224, 384       # 1920x1080 → 384x224 (CPU용). 32의 배수 → U-Net(÷16), U-Net++(÷32) 모두 가능
BASE_CH      = 16             # U-Net 첫 층 채널 수 (GPU라면 32 권장)
NUM_CLASSES  = 11             # 팔레트 탐지 후 실제 클래스 수로 재확인
IGNORE_INDEX = 255            # 팔레트에 속하지 않는 픽셀(경계 안티앨리어싱 등)은 손실/지표에서 제외
NUM_WORKERS  = 0              # Windows + Jupyter에서는 0이 안전 (멀티프로세스 로더 멈춤 방지)

EPOCHS_MAIN  = 5 if QUICK_MODE else 40    # 본 실험 최대 에폭 (조기 종료 포함)
EPOCHS_TUNE  = 3 if QUICK_MODE else 12    # 하이퍼파라미터 탐색용 짧은 학습
PATIENCE     = 10                          # Val mIoU가 10 에폭 동안 개선되지 않으면 조기 종료

# 실행할 실험 선택 (CPU 시간이 부족하면 False로)
RUN_TUNING = True    # E4 그리드 탐색 (False면 lr=1e-3, batch=4 사용)
RUN_E5     = True    # Attention U-Net
RUN_E6     = True    # U-Net++ (사전학습 인코더, 가중치 다운로드 필요)

SAVE_DIR = "outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. 데이터 다운로드 및 파일 구조 확인

`kagglehub.dataset_download()`로 데이터를 내려받고, 파일 확장자·이름 패턴을 출력해 구조를 파악합니다.
이 데이터셋은 원본 프레임(`Frame 1  (1).jpg`)마다 색상 코드 마스크(`...jpg___save.png`)와 원본 위에 마스크를 겹친 이미지(`...jpg___fuse.png`)가 함께 제공되는 형태로 알려져 있습니다. 학습 정답으로는 **순수 마스크인 `___save.png`** 를 사용합니다.

> `kagglehub`가 실패하면: Kaggle 계정 → Settings → API → `kaggle.json` 발급 후
> `!kaggle datasets download -d sadhliroomyprime/football-semantic-segmentation --unzip -p data` 로 받고 `DATA_ROOT = "data"` 로 지정하세요.

In [ ]:
# ===== 1-1. 데이터 다운로드 =====
import kagglehub
DATA_ROOT = kagglehub.dataset_download("sadhliroomyprime/football-semantic-segmentation")
print("DATA_ROOT:", DATA_ROOT)

# 전체 파일 목록과 확장자 분포
all_files = sorted(f for f in glob.glob(os.path.join(DATA_ROOT, "**", "*"), recursive=True) if os.path.isfile(f))
print("파일 수:", len(all_files))
print("확장자 분포:", Counter(os.path.splitext(f)[1].lower() for f in all_files))
print("\n파일 예시:")
for f in all_files[:12]:
    print("  ", os.path.relpath(f, DATA_ROOT))

In [ ]:
# ===== 1-2. 이미지 ↔ 마스크 짝 맞추기 =====
IMG_EXT = (".jpg", ".jpeg", ".png")

def pair_save_pattern(files):
    """패턴 A: 'xxx.jpg' ↔ 'xxx.jpg___save.png' (같은 폴더)"""
    fs = set(files)
    imgs = [f for f in files if f.lower().endswith(IMG_EXT) and "___" not in os.path.basename(f)]
    return [(p, p + "___save.png") for p in imgs if p + "___save.png" in fs]

def pair_mask_folder(files):
    """패턴 B(대체): 'images/xxx.jpg' ↔ 'masks/xxx.png' (파일명 stem 일치)"""
    masks = [f for f in files if "mask" in f.lower() and f.lower().endswith(IMG_EXT)]
    stem = lambda p: os.path.splitext(os.path.basename(p))[0]
    mask_map = {stem(m): m for m in masks}
    imgs = [f for f in files if f.lower().endswith(IMG_EXT) and "mask" not in f.lower() and "___" not in f]
    return [(p, mask_map[stem(p)]) for p in imgs if stem(p) in mask_map]

pairs = pair_save_pattern(all_files) or pair_mask_folder(all_files)
pairs = sorted(pairs)
print("이미지-마스크 쌍:", len(pairs))
assert len(pairs) > 0, "짝을 찾지 못했습니다. 위 '파일 예시'를 보고 pair 함수를 수정하세요."
for p, m in pairs[:3]:
    print("  IMG :", os.path.basename(p), "\n  MASK:", os.path.basename(m))

## 2. EDA — 색상 팔레트 → 클래스 매핑

마스크는 클래스마다 고유한 **RGB 색**으로 칠해진 이미지입니다. 모델은 정수 라벨(0~10)이 필요하므로 "어떤 색 = 어떤 클래스"인지 확정해야 합니다.
색상표를 하드코딩하면 데이터 버전에 따라 틀릴 위험이 있어, 다음 우선순위로 **데이터에서 자동으로** 팔레트를 결정합니다.

1. `MANUAL_CLASS_COLORS` — 직접 지정한 색상표(있다면 최우선)
2. 데이터에 포함된 **라벨-색상 CSV**
3. 데이터에 포함된 **COCO JSON**: 각 카테고리 폴리곤 내부에서 가장 많이 나오는 마스크 색을 찾고, 헝가리안 알고리즘(`linear_sum_assignment`)으로 카테고리↔색을 1:1 매칭
4. 위가 모두 없으면: 픽셀 빈도 상위 11개 색을 클래스로 사용(이름은 `class_k`)

**관전 포인트**: 상위 11개 색 이후에도 비율이 무시할 수 없는 색이 남는지(=라벨 노이즈/안티앨리어싱) 확인합니다.

In [ ]:
# ===== 2-1. 마스크 전체의 색상 빈도 집계 =====
def read_rgb(path):
    # cv2.imread는 Windows에서 한글·특수문자 경로를 읽지 못하므로 바이트로 읽은 뒤 디코딩
    bgr = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        raise IOError(f"이미지를 읽을 수 없습니다: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def rgb_to_code(arr):            # (...,3) uint8 → 24bit 정수 코드 (색 비교를 빠르게)
    a = arr.astype(np.int64)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

def code_to_rgb(c):
    return ((c >> 16) & 255, (c >> 8) & 255, c & 255)

color_counter = Counter()
for _, m in tqdm(pairs, desc="mask colors"):
    u, c = np.unique(rgb_to_code(read_rgb(m)).ravel(), return_counts=True)
    color_counter.update(dict(zip(u.tolist(), c.tolist())))

total_px = sum(color_counter.values())
df_colors = pd.DataFrame(
    [(code_to_rgb(k), v, v / total_px * 100) for k, v in color_counter.most_common(20)],
    columns=["RGB", "pixels", "ratio(%)"])
print("서로 다른 색 수:", len(color_counter))
print(f"상위 11색 누적 비율: {df_colors['ratio(%)'][:11].sum():.3f}%")
df_colors

In [ ]:
# ===== 2-2. 팔레트(색→클래스) 결정 =====
MANUAL_CLASS_COLORS = None   # 예: {"Ground": (r,g,b), "Ball": (r,g,b), ...}  필요 시 직접 지정

def palette_from_csv(files):
    """라벨-색상 CSV가 있으면 (이름, RGB) 목록을 반환"""
    for f in files:
        if not f.lower().endswith(".csv"):
            continue
        df = pd.read_csv(f)
        name_col = df.select_dtypes(exclude="number").columns
        if len(name_col) == 0:
            continue
        num = df.select_dtypes(include="number")
        out = []
        for i, row in df.iterrows():
            if num.shape[1] >= 3:
                rgb = tuple(int(v) for v in num.iloc[i, :3])
            else:   # "(r, g, b)" 또는 "#rrggbb" 형태 문자열 파싱
                s = " ".join(str(v) for v in row.drop(name_col[0]).values)
                h = re.search(r"#([0-9a-fA-F]{6})", s)
                nums = re.findall(r"\d+", s)
                if h:   rgb = tuple(int(h.group(1)[k:k+2], 16) for k in (0, 2, 4))
                elif len(nums) >= 3: rgb = tuple(int(n) for n in nums[:3])
                else:   continue
            out.append((str(row[name_col[0]]).strip(), rgb))
        if len(out) >= 2:
            print("CSV 팔레트 사용:", os.path.basename(f))
            return out
    return None

def palette_from_coco(files, pairs, top_k=30, max_imgs=40):
    """COCO 폴리곤 영역 안의 마스크 색 분포로 카테고리↔색 1:1 매칭"""
    jsons = [f for f in files if f.lower().endswith(".json")]
    if not jsons:
        return None
    coco = json.load(open(jsons[0]))
    if not {"categories", "annotations", "images"} <= set(coco):
        return None
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    cat_ids = sorted(cats)
    imgs = {im["id"]: im for im in coco["images"]}
    mask_by_name = {os.path.basename(p): m for p, m in pairs}
    cand = [k for k, _ in color_counter.most_common(top_k)]
    cidx = {c: i for i, c in enumerate(cand)}
    cnt = np.zeros((len(cat_ids), len(cand)))
    anns_by_img = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_img[a["image_id"]].append(a)
    used = 0
    for img_id, anns in anns_by_img.items():
        im = imgs.get(img_id)
        m = mask_by_name.get(os.path.basename(im["file_name"])) if im else None
        if m is None:
            continue
        codes = rgb_to_code(read_rgb(m)); H, W = codes.shape
        sx, sy = W / im.get("width", W), H / im.get("height", H)
        for a in anns:
            seg = a.get("segmentation")
            if not isinstance(seg, list):
                continue
            pm = np.zeros((H, W), np.uint8)
            for poly in seg:
                pts = np.array(poly, dtype=np.float64).reshape(-1, 2) * [sx, sy]
                cv2.fillPoly(pm, [pts.round().astype(np.int32)], 1)
            u, c = np.unique(codes[pm == 1], return_counts=True)
            r = cat_ids.index(a["category_id"])
            for uu, cc in zip(u.tolist(), c.tolist()):
                if uu in cidx:
                    cnt[r, cidx[uu]] += cc
        used += 1
        if used >= max_imgs:
            break
    rows = np.where(cnt.sum(1) > 0)[0]
    if len(rows) < 2:
        return None
    frac = cnt[rows] / cnt[rows].sum(1, keepdims=True)
    r_ind, c_ind = linear_sum_assignment(-frac)      # 비율 합 최대화
    print(f"COCO 매칭 사용: {os.path.basename(jsons[0])} (이미지 {used}장 기준)")
    for r, c in zip(r_ind, c_ind):
        print(f"   {cats[cat_ids[rows[r]]]:<22s} → RGB{code_to_rgb(cand[c])}  (폴리곤 내 비율 {frac[r, c]*100:.1f}%)")
    return [(cats[cat_ids[rows[r]]], code_to_rgb(cand[c])) for r, c in zip(r_ind, c_ind)]

if MANUAL_CLASS_COLORS:
    named = list(MANUAL_CLASS_COLORS.items()); src = "manual"
else:
    named = palette_from_csv(all_files); src = "csv"
    if not named:
        named = palette_from_coco(all_files, pairs); src = "coco"
    if not named:
        named = [(f"class_{i}", code_to_rgb(k)) for i, (k, _) in enumerate(color_counter.most_common(11))]
        src = "frequency"

# 픽셀 빈도 순으로 정렬 → 인덱스 0이 가장 큰 클래스
freq = {code_to_rgb(k): v for k, v in color_counter.items()}
named = sorted(named, key=lambda x: -freq.get(tuple(x[1]), 0))
CLASS_NAMES = [n for n, _ in named]
PALETTE = np.array([c for _, c in named], dtype=np.uint8)
NUM_CLASSES = len(CLASS_NAMES)
print(f"\n팔레트 출처: {src} | 클래스 수: {NUM_CLASSES}")
if NUM_CLASSES != 11:
    print("⚠️ 클래스 수가 11이 아닙니다. 위 색상 표를 보고 MANUAL_CLASS_COLORS로 지정하세요.")
for i, (n, c) in enumerate(named):
    print(f"  {i:2d} {n:<22s} RGB{tuple(int(v) for v in c)}  비율 {freq.get(tuple(c),0)/total_px*100:6.2f}%")

In [ ]:
# ===== 2-3. 원본 / 마스크 / 범례 시각화 =====
def show_legend(ax):
    for i, (n, c) in enumerate(zip(CLASS_NAMES, PALETTE)):
        ax.add_patch(plt.Rectangle((0, i), 1, 0.8, color=c / 255))
        ax.text(1.2, i + 0.4, f"{i}: {n}", va="center", fontsize=9)
    ax.set_xlim(0, 6); ax.set_ylim(NUM_CLASSES, -0.2); ax.axis("off")

fig, axes = plt.subplots(2, 3, figsize=(18, 7), gridspec_kw={"width_ratios": [4, 4, 1.6]})
for r, idx in enumerate([0, len(pairs) // 2]):
    p, m = pairs[idx]
    axes[r, 0].imshow(read_rgb(p)); axes[r, 0].set_title(os.path.basename(p)); axes[r, 0].axis("off")
    axes[r, 1].imshow(read_rgb(m)); axes[r, 1].set_title("color mask (___save.png)"); axes[r, 1].axis("off")
    show_legend(axes[r, 2])
plt.tight_layout(); plt.show()
print("원본 해상도:", read_rgb(pairs[0][0]).shape)

## 3. 전처리

### 3-1. RGB 마스크 → 클래스 인덱스 맵
- 각 픽셀 색을 팔레트에서 **가장 가까운 색**(유클리드 거리)으로 매핑합니다. 경계의 안티앨리어싱 색도 흡수됩니다.
- 거리가 `max_dist`(30)를 넘는 색은 어떤 클래스로도 보기 어려우므로 `IGNORE_INDEX(255)`로 표시해 손실·지표에서 제외합니다.
- 계산 효율: 픽셀 200만 개를 직접 비교하지 않고 **고유 색만** 거리 계산 후 역인덱스로 펼칩니다.

### 3-2. 리사이즈
- 원본(1920×1080)을 **384×224**로 줄입니다(CPU 학습 시간 고려). 이미지는 `INTER_AREA`(축소 시 앨리어싱 최소), 마스크는 반드시 `INTER_NEAREST`(라벨 값이 섞이면 안 됨).
- 트레이드오프: 해상도를 줄이면 학습이 빨라지지만 **공(Ball)·골대(Goal Bar)처럼 가는/작은 객체가 수 픽셀로 축소**됩니다 → 이 클래스들의 IoU가 낮게 나올 것으로 예상합니다.
- 데이터가 약 100장이라 전부 메모리에 올려두고(수십 MB) 매 에폭 디스크 I/O를 피합니다.

In [ ]:
# ===== 3-1. 마스크 변환 함수 및 전체 데이터 적재 =====
def rgb_mask_to_index(rgb, palette=PALETTE, max_dist=30):
    codes = rgb_to_code(rgb).ravel()
    u, inv = np.unique(codes, return_inverse=True)                 # 고유 색만 처리
    ucol = np.stack([(u >> 16) & 255, (u >> 8) & 255, u & 255], 1).astype(np.int32)
    d2 = ((ucol[:, None, :] - palette[None].astype(np.int32)) ** 2).sum(-1)
    idx = d2.argmin(1).astype(np.uint8)
    idx[np.sqrt(d2.min(1)) > max_dist] = IGNORE_INDEX              # 팔레트와 먼 색 → 무시
    return idx[inv].reshape(rgb.shape[:2])

X_all, Y_all, NAMES = [], [], []
for p, m in tqdm(pairs, desc="load & resize"):
    img = cv2.resize(read_rgb(p), (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    lab = rgb_mask_to_index(read_rgb(m))
    lab = cv2.resize(lab, (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
    X_all.append(img); Y_all.append(lab); NAMES.append(os.path.basename(p))
X_all, Y_all = np.stack(X_all), np.stack(Y_all)
print("X:", X_all.shape, X_all.dtype, "| Y:", Y_all.shape, Y_all.dtype)
print(f"IGNORE 픽셀 비율: {(Y_all == IGNORE_INDEX).mean()*100:.3f}%")

### 3-3. 클래스 분포 분석

**가설**: 그라운드가 픽셀의 절반 이상을 차지하고, 공·골대·심판·골키퍼는 1% 미만일 것입니다.
**관전 포인트**: (1) 픽셀 비율이 낮은 클래스가 이후 클래스별 IoU에서도 낮게 나오는지(상관관계), (2) "이미지 등장률"이 낮은 클래스(예: Goal Bar, 골키퍼)는 테스트 세트에 아예 없을 수도 있어 IoU가 `NaN`으로 표시될 수 있습니다.

In [ ]:
# ===== 3-3. 클래스별 픽셀 비율 / 이미지 등장률 =====
valid = Y_all[Y_all != IGNORE_INDEX]
px_ratio = np.bincount(valid, minlength=NUM_CLASSES) / valid.size * 100
img_ratio = np.array([(Y_all == c).any(axis=(1, 2)).mean() * 100 for c in range(NUM_CLASSES)])
df_dist = pd.DataFrame({"class": CLASS_NAMES, "pixel_%": px_ratio.round(3), "images_%": img_ratio.round(1)})
display(df_dist)

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].bar(CLASS_NAMES, px_ratio, color=PALETTE / 255, edgecolor="k"); ax[0].set_yscale("log")
ax[0].set_title("Pixel ratio per class (%) - log scale"); ax[0].tick_params(axis="x", rotation=60)
ax[1].bar(CLASS_NAMES, img_ratio, color=PALETTE / 255, edgecolor="k")
ax[1].set_title("Images containing class (%)"); ax[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

### 3-4. Train / Validation / Test 분할 (70 / 15 / 15)

- **Train**: 가중치 학습 / **Validation**: 조기 종료·하이퍼파라미터·모델 선택 / **Test**: 최종 성능 보고 전용(선택에 쓰지 않음).
- 시드 고정 무작위 분할을 사용합니다.
- **관전 포인트(한계)**: 프레임들이 같은 경기 영상에서 추출되어 인접 프레임이 매우 비슷할 수 있습니다. 무작위 분할 시 비슷한 장면이 train/test 양쪽에 들어가 **테스트 성능이 실제보다 낙관적**일 수 있습니다.

In [ ]:
# ===== 3-4. 데이터 분할 =====
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(X_all))
n_tr, n_va = int(len(perm) * 0.70), int(len(perm) * 0.15)
idx_tr, idx_va, idx_te = perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]
X_tr, Y_tr = X_all[idx_tr], Y_all[idx_tr]
X_va, Y_va = X_all[idx_va], Y_all[idx_va]
X_te, Y_te = X_all[idx_te], Y_all[idx_te]
print(f"train {len(idx_tr)} | val {len(idx_va)} | test {len(idx_te)}")

# 각 분할에 클래스가 몇 장씩 등장하는지 (테스트에 없는 클래스는 IoU 계산 불가)
presence = pd.DataFrame({s: [(Y == c).any(axis=(1, 2)).sum() for c in range(NUM_CLASSES)]
                         for s, Y in [("train", Y_tr), ("val", Y_va), ("test", Y_te)]}, index=CLASS_NAMES)
presence

## 4. Dataset 클래스와 데이터 증강

### 정규화
ImageNet 평균/표준편차로 정규화합니다. 직접 구현한 U-Net은 어떤 값이든 상관없지만, E6의 **사전학습 인코더**는 이 통계로 학습되었으므로 모든 실험에 동일하게 적용해 조건을 맞춥니다.

### 증강 기법 (학습 세트에만, 이미지와 마스크에 **동일한 기하 변환** 적용)
| 기법 | 구현 | 의도 |
|---|---|---|
| Random Resized Crop | 원본의 60~100% 영역을 잘라 원래 크기로 확대 (p=0.7) | 카메라 줌·거리 변화 모사, 작은 객체를 크게 보는 효과 |
| Horizontal Flip | 좌우 반전 (p=0.5) | 경기장 좌/우 방향 다양화 (축구 장면은 좌우 대칭이 자연스러움) |
| Color Jitter | 밝기·대비·채도 ±30%, 색조 ±0.05 (p=0.8, **이미지에만**) | 조명·중계 카메라·날씨 차이에 강건하게 |

- 마스크는 크롭 확대 시 **최근접 보간**으로 라벨 값을 보존합니다.
- **관전 포인트**: 팀 A/B와 골키퍼는 **유니폼 색**이 핵심 단서입니다. 색조(hue)를 크게 흔들면 팀 구분 단서를 망가뜨릴 수 있어 hue를 ±0.05로 작게 잡았습니다. 증강 후 Team A/B IoU가 오히려 떨어지는지 확인해 보세요.

In [ ]:
# ===== 4-1. Dataset 정의 =====
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

class FootballDataset(Dataset):
    def __init__(self, images, masks, augment=False):
        self.images, self.masks, self.augment = images, masks, augment
        self.color_jitter = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05)

    def __len__(self):
        return len(self.images)

    def _augment(self, img, mask):
        H, W = img.shape[1:]
        if random.random() < 0.7:                                   # Random Resized Crop (종횡비 유지)
            s = random.uniform(0.6, 1.0)
            ch, cw = int(H * s), int(W * s)
            top, left = random.randint(0, H - ch), random.randint(0, W - cw)
            img = TF.resized_crop(img, top, left, ch, cw, [H, W], antialias=True)
            mask = TF.resized_crop(mask[None].float(), top, left, ch, cw, [H, W],
                                   interpolation=T.InterpolationMode.NEAREST)[0].long()
        if random.random() < 0.5:                                   # Horizontal Flip
            img, mask = TF.hflip(img), TF.hflip(mask)
        if random.random() < 0.8:                                   # Color Jitter (이미지만)
            img = self.color_jitter(img)
        return img.clamp(0, 1), mask

    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i]).permute(2, 0, 1).float() / 255.0   # (3,H,W) 0~1
        mask = torch.from_numpy(self.masks[i].astype(np.int64))                     # (H,W) 정수 라벨
        if self.augment:
            img, mask = self._augment(img, mask)
        return TF.normalize(img, MEAN, STD), mask

def denorm(t):   # 시각화용 정규화 해제
    return (t.permute(1, 2, 0).cpu().numpy() * STD + MEAN).clip(0, 1)

def colorize(lab):  # 인덱스 맵 → RGB (IGNORE는 검정)
    out = np.zeros((*lab.shape, 3), np.uint8)
    ok = lab != IGNORE_INDEX
    out[ok] = PALETTE[lab[ok]]
    return out

In [ ]:
# ===== 4-2. 증강 결과 확인 (같은 이미지에 4번 적용) =====
ds_aug = FootballDataset(X_tr, Y_tr, augment=True)
fig, axes = plt.subplots(2, 4, figsize=(20, 6))
for k in range(4):
    x, y = ds_aug[0]
    axes[0, k].imshow(denorm(x)); axes[0, k].set_title(f"augmented #{k+1}"); axes[0, k].axis("off")
    axes[1, k].imshow(colorize(y.numpy())); axes[1, k].axis("off")
plt.tight_layout(); plt.show()

## 5. 모델 구현

### 5-1. U-Net (Ronneberger et al., 2015)
```
입력(3) ─ Enc1(16) ──────────────────────────── skip ─→ Dec1(16) ─ 1x1 Conv → 11 클래스 logits
           └ pool ─ Enc2(32) ───────────── skip ─→ Dec2(32) ┘
                     └ pool ─ Enc3(64) ──── skip ─→ Dec3(64) ┘
                               └ pool ─ Enc4(128) ─ skip ─→ Dec4(128) ┘
                                         └ pool ─ Bottleneck(256) ─ up ┘
```
- **인코더(수축 경로)**: `DoubleConv`(3×3 Conv → BN → ReLU ×2) + MaxPool로 해상도를 1/2씩 줄이며 "무엇인지"(의미)를 학습
- **디코더(확장 경로)**: `ConvTranspose2d`로 2배 업샘플 → 같은 해상도의 인코더 특징과 **채널 방향 concat(skip connection)** → `DoubleConv`. skip이 잃어버린 위치·경계 정보("어디인지")를 복원합니다.
- 원 논문 대비 변경: padding=1로 입력·출력 크기를 같게 유지, BatchNorm 추가(학습 안정), 기본 채널 64→**16**(CPU 연산량 절감 + 데이터 70장에 맞춘 파라미터 축소로 과적합 완화)

### 5-2. Attention U-Net (Oktay et al., 2018)
skip 특징을 디코더에 그냥 붙이지 않고 **Attention Gate**를 통과시킵니다.
- 게이트 신호 `g`(디코더 특징, "지금 무엇을 찾는지")와 skip 특징 `x`를 각각 1×1 Conv → 더하기 → ReLU → 1×1 Conv → Sigmoid → **픽셀별 가중치 α∈(0,1)**
- `x · α`만 디코더로 전달 → 관중·광고판 같은 배경 텍스처는 억제하고 관련 영역만 강조
- 파라미터 증가는 매우 작습니다(아래 출력으로 확인).

In [ ]:
# ===== 5. U-Net / Attention U-Net 정의 =====
class DoubleConv(nn.Sequential):
    """(3x3 Conv → BN → ReLU) × 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

class AttentionGate(nn.Module):
    """g: 디코더(게이트) 특징, x: 인코더 skip 특징 → x * alpha"""
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(g_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch))
        self.W_x = nn.Sequential(nn.Conv2d(x_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch))
        self.psi = nn.Sequential(nn.Conv2d(inter_ch, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        alpha = self.psi(F.relu(self.W_g(g) + self.W_x(x), inplace=True))   # (B,1,H,W)
        return x * alpha

class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=11, base=BASE_CH, attention=False):
        super().__init__()
        chs = [base * 2 ** i for i in range(5)]                 # base=16 → [16, 32, 64, 128, 256]
        self.attention = attention
        self.inc = DoubleConv(in_ch, chs[0])
        self.downs = nn.ModuleList([nn.Sequential(nn.MaxPool2d(2), DoubleConv(chs[i], chs[i + 1]))
                                    for i in range(4)])
        self.ups, self.gates, self.decs = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for i in range(4, 0, -1):                               # 256→128, 128→64, ...
            self.ups.append(nn.ConvTranspose2d(chs[i], chs[i - 1], 2, stride=2))
            self.gates.append(AttentionGate(chs[i - 1], chs[i - 1], chs[i - 1] // 2) if attention else nn.Identity())
            self.decs.append(DoubleConv(chs[i - 1] * 2, chs[i - 1]))
        self.head = nn.Conv2d(chs[0], n_classes, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for down in self.downs:
            skips.append(down(skips[-1]))
        x = skips.pop()                                          # bottleneck
        for up, gate, dec in zip(self.ups, self.gates, self.decs):
            x = up(x)
            s = skips.pop()
            if self.attention:
                s = gate(g=x, x=s)
            x = dec(torch.cat([s, x], dim=1))                    # skip connection
        return self.head(x)                                      # (B, n_classes, H, W) logits

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

for name, m in [("U-Net", UNet(n_classes=NUM_CLASSES)), ("Attention U-Net", UNet(n_classes=NUM_CLASSES, attention=True))]:
    out = m(torch.randn(1, 3, IMG_H, IMG_W))
    print(f"{name:16s} params {count_params(m):.2f}M | 출력 shape {tuple(out.shape)}")

## 6. 손실 함수와 평가 지표

### 손실 함수
- **Cross Entropy (CE)**: 픽셀마다 정답 클래스 확률의 −log를 평균. 안정적이고 수렴이 빠르지만, 픽셀 수가 압도적인 그라운드에 손실이 지배되어 **소수 클래스를 무시하는 경향**이 있습니다.
- **Dice Loss**: 클래스 c마다 소프트 Dice 계수 `2·Σ(p·g) / (Σp + Σg)`를 계산하고 `1 − 평균`을 손실로 씁니다. 클래스 크기와 무관하게 **겹침 비율**을 보므로 작은 객체에도 동일한 비중을 줍니다. 배치에 **실제로 존재하는 클래스만** 평균해 불안정을 줄였습니다.
- **CE + Dice**: CE의 안정적 그래디언트 + Dice의 클래스 균형 효과를 결합 (Segmentation에서 널리 쓰는 조합)

### 평가 지표 (테스트 세트 전체 픽셀을 모은 **혼동행렬** 기반)
| 지표 | 정의 | 의미 |
|---|---|---|
| Pixel Accuracy | 맞힌 픽셀 / 전체 픽셀 | 큰 클래스에 지배됨 — 참고용 |
| IoU (클래스별) | TP / (TP + FP + FN) | 예측 영역과 정답 영역의 겹침 |
| **mIoU** | 클래스별 IoU 평균 | **주 지표** (모든 클래스를 동등하게) |
| Dice (클래스별/평균) | 2TP / (2TP + FP + FN) | IoU와 단조 관계, 의료·세그멘테이션 표준 지표 |
| CE Loss / Dice Loss | 테스트 세트에서의 손실 값 | 미션 요구 지표 — 확률 보정 정도까지 반영 |

테스트에 등장하지 않는 클래스는 IoU를 `NaN`으로 두고 평균에서 제외합니다.

In [ ]:
# ===== 6. 손실 함수 & 지표 =====
class DiceLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=IGNORE_INDEX, smooth=1.0):
        super().__init__()
        self.C, self.ignore, self.smooth = num_classes, ignore_index, smooth

    def forward(self, logits, target):
        probs = logits.float().softmax(dim=1)                          # (B,C,H,W)
        valid = (target != self.ignore)                                # 무시 픽셀 제외
        t = torch.where(valid, target, torch.zeros_like(target))
        onehot = F.one_hot(t, self.C).permute(0, 3, 1, 2).float()
        v = valid.unsqueeze(1).float()
        probs, onehot = probs * v, onehot * v
        dims = (0, 2, 3)
        inter = (probs * onehot).sum(dims)
        card = probs.sum(dims) + onehot.sum(dims)
        dice = (2 * inter + self.smooth) / (card + self.smooth)       # 클래스별 soft Dice
        present = onehot.sum(dims) > 0                                 # 배치에 존재하는 클래스만
        return 1 - dice[present].mean()

ce_fn   = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
dice_fn = DiceLoss(NUM_CLASSES)

def build_criterion(loss_type):
    if loss_type == "ce":
        return lambda logits, y: ce_fn(logits.float(), y)
    if loss_type == "ce_dice":
        return lambda logits, y: ce_fn(logits.float(), y) + dice_fn(logits, y)
    raise ValueError(loss_type)

def metrics_from_cm(cm):
    tp = np.diag(cm).astype(np.float64)
    gt, pr = cm.sum(1), cm.sum(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        iou  = np.where(gt > 0, tp / (gt + pr - tp), np.nan)        # 정답에 없는 클래스 = NaN
        dice = np.where(gt > 0, 2 * tp / (gt + pr), np.nan)
    return {"pixel_acc": tp.sum() / cm.sum(), "miou": np.nanmean(iou), "mdice": np.nanmean(dice),
            "iou": iou, "dice": dice, "cm": cm}

## 7. 학습 · 평가 루프

- **Optimizer**: AdamW (가중치 감쇠 1e-4) — Adam의 적응적 학습률 + 올바른 L2 규제
- **Scheduler**: CosineAnnealing — 학습률을 코사인 곡선으로 0 근처까지 감소시켜 후반에 미세 조정
- **AMP**: GPU에서 fp16 혼합 정밀도로 속도↑·메모리↓ (손실은 fp32로 계산해 수치 안정)
- **체크포인트 & 조기 종료**: 매 에폭 Val mIoU를 측정해 최고일 때만 저장, `PATIENCE` 에폭 동안 개선이 없으면 종료. 학습이 끝나면 **최고 Val 시점의 가중치**를 다시 불러옵니다.
- `run_experiment()`는 학습 → 최고 체크포인트 로드 → Val/Test 평가 → 결과를 `RESULTS`에 기록 → 학습 곡선 출력까지 한 번에 수행합니다.

In [ ]:
# ===== 7. 학습/평가 함수 =====
val_dl  = DataLoader(FootballDataset(X_va, Y_va), batch_size=4, shuffle=False, num_workers=NUM_WORKERS)
test_dl = DataLoader(FootballDataset(X_te, Y_te), batch_size=4, shuffle=False, num_workers=NUM_WORKERS)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    C = NUM_CLASSES
    cm = torch.zeros(C * C, dtype=torch.int64, device=DEVICE)
    ce_sum = dice_sum = n = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(x)
        logits = logits.float()
        ce_sum += ce_fn(logits, y).item() * x.size(0)
        dice_sum += dice_fn(logits, y).item() * x.size(0)
        n += x.size(0)
        pred = logits.argmax(1)
        ok = y != IGNORE_INDEX
        cm += torch.bincount(y[ok] * C + pred[ok], minlength=C * C)   # 혼동행렬 누적
    m = metrics_from_cm(cm.view(C, C).cpu().numpy())
    m.update(ce_loss=ce_sum / n, dice_loss=dice_sum / n)
    return m

def train_model(model, name, epochs, lr, batch_size, loss_type, augment, weight_decay=1e-4,
                patience=PATIENCE, verbose=True):
    model = model.to(DEVICE)
    train_dl = DataLoader(FootballDataset(X_tr, Y_tr, augment=augment), batch_size=batch_size,
                          shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=USE_AMP)
    criterion = build_criterion(loss_type)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    ckpt = os.path.join(SAVE_DIR, f"{name}.pt")
    best, wait, hist, t0 = -1.0, 0, [], time.time()

    for ep in range(1, epochs + 1):
        model.train(); run, cnt = 0.0, 0
        for x, y in train_dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(x)
            loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * x.size(0); cnt += x.size(0)
        sched.step()

        vm = evaluate(model, val_dl)
        val_loss = vm["ce_loss"] + (vm["dice_loss"] if loss_type == "ce_dice" else 0)
        hist.append(dict(epoch=ep, train_loss=run / cnt, val_loss=val_loss,
                         val_miou=vm["miou"], val_pixel_acc=vm["pixel_acc"], lr=opt.param_groups[0]["lr"]))
        if vm["miou"] > best:
            best, wait = vm["miou"], 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
        if verbose and (ep == 1 or ep % 10 == 0):
            print(f"[{name}] ep {ep:3d} | train {run/cnt:.4f} | val {val_loss:.4f} "
                  f"| val mIoU {vm['miou']:.4f} | best {best:.4f}")
        if wait >= patience:
            if verbose: print(f"[{name}] 조기 종료 @ ep {ep} (best val mIoU {best:.4f})")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, pd.DataFrame(hist), time.time() - t0

def plot_history(hist, name):
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
    ax[0].plot(hist.epoch, hist.train_loss, label="train loss")
    ax[0].plot(hist.epoch, hist.val_loss, label="val loss"); ax[0].legend(); ax[0].set_title(f"{name} — Loss")
    ax[1].plot(hist.epoch, hist.val_miou, label="val mIoU", color="tab:green")
    b = hist.val_miou.idxmax()
    ax[1].scatter(hist.epoch[b], hist.val_miou[b], color="red", zorder=3, label=f"best ep {hist.epoch[b]}")
    ax[1].legend(); ax[1].set_title(f"{name} — Val mIoU")
    plt.tight_layout(); plt.show()

RESULTS = {}

# E6 U-Net++ 설정 (CPU용: ResNet18 인코더 + 가벼운 디코더)
E6_KW = dict(encoder_name="resnet18", in_channels=3, classes=NUM_CLASSES,
             decoder_channels=(128, 64, 32, 16, 16))

def run_experiment(name, model_fn, epochs, lr, batch_size, loss_type, augment, arch):
    seed_everything(SEED)
    model = model_fn()
    model, hist, sec = train_model(model, name, epochs, lr, batch_size, loss_type, augment)
    val_m, test_m = evaluate(model, val_dl), evaluate(model, test_dl)
    RESULTS[name] = dict(arch=arch, loss=loss_type, aug=augment, lr=lr, bs=batch_size,
                         epochs_run=len(hist), best_epoch=int(hist.epoch[hist.val_miou.idxmax()]),
                         params_M=count_params(model), minutes=sec / 60,
                         hist=hist, val=val_m, test=test_m)
    summary_table().to_csv(os.path.join(SAVE_DIR, "results_summary.csv"))   # 중간 저장(커널 중단 대비)
    hist.to_csv(os.path.join(SAVE_DIR, f"history_{name}.csv"), index=False)
    print(f"\n▶ {name} | Val mIoU {val_m['miou']:.4f} | Test mIoU {test_m['miou']:.4f} "
          f"| Test mDice {test_m['mdice']:.4f} | Test PixAcc {test_m['pixel_acc']:.4f} "
          f"| Test CE {test_m['ce_loss']:.4f} | Test DiceLoss {test_m['dice_loss']:.4f} | {sec/60:.1f} min")
    plot_history(hist, name)
    return model

def summary_table():
    rows = []
    for k, r in RESULTS.items():
        rows.append(dict(exp=k, arch=r["arch"], loss=r["loss"], aug=r["aug"], lr=r["lr"], bs=r["bs"],
                         epochs_run=r["epochs_run"], best_ep=r["best_epoch"], params_M=round(r["params_M"], 2),
                         val_mIoU=r["val"]["miou"], test_mIoU=r["test"]["miou"], test_mDice=r["test"]["mdice"],
                         test_PixAcc=r["test"]["pixel_acc"], test_CE=r["test"]["ce_loss"],
                         test_DiceLoss=r["test"]["dice_loss"], minutes=round(r["minutes"], 1)))
    return pd.DataFrame(rows).set_index("exp").round(4)

def per_class_table(split="test"):
    return pd.DataFrame({k: r[split]["iou"] for k, r in RESULTS.items()}, index=CLASS_NAMES).T.round(3)

### 7-1. CPU 속도 측정 → 예상 소요 시간

본 실험 전에 학습 3스텝을 실제로 돌려 **에폭당 시간**을 재고, 현재 설정 기준 전체 예상 시간을 계산합니다.
예상 시간이 너무 길면 0-2 셀에서 `RUN_TUNING`/`RUN_E5`/`RUN_E6`를 끄거나 `EPOCHS_MAIN`을 줄인 뒤 0-2 셀부터 다시 실행하세요. (조기 종료가 있어 실제로는 예상보다 짧게 끝나는 경우가 많습니다.)

In [ ]:
# ===== 7-1. 속도 측정 =====
def time_per_epoch(model_fn, batch_size=4, steps=3):
    seed_everything(SEED)
    model = model_fn().to(DEVICE); model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    crit = build_criterion("ce_dice")
    dl = DataLoader(FootballDataset(X_tr, Y_tr, augment=True), batch_size=batch_size, shuffle=True,
                    drop_last=True, num_workers=NUM_WORKERS)
    it = iter(dl); x, y = next(it)
    crit(model(x.to(DEVICE)), y.to(DEVICE)).backward()               # 워밍업 1스텝
    t0 = time.time()
    for _ in range(steps):
        x, y = next(it); opt.zero_grad()
        crit(model(x.to(DEVICE)), y.to(DEVICE)).backward(); opt.step()
    per_step = (time.time() - t0) / steps
    train_sec = per_step * (len(X_tr) // batch_size)
    eval_sec = per_step / 3 * (len(X_va) / batch_size)                # 평가(순전파만) ≈ 학습 스텝의 1/3
    return train_sec + eval_sec

t_unet = time_per_epoch(lambda: UNet(n_classes=NUM_CLASSES))
print(f"U-Net 1 에폭 ≈ {t_unet:.1f}초")
total = t_unet * EPOCHS_MAIN * 3                                      # E1, E2, E3
total += t_unet * EPOCHS_MAIN                                         # E4
if RUN_TUNING: total += t_unet * EPOCHS_TUNE * 4 * 0.8               # 4개 조합 (bs 8은 스텝 수 절반)
if RUN_E5:     total += t_unet * 1.15 * EPOCHS_MAIN                   # Attention Gate 오버헤드 소폭
print(f"E1~E5 최대 예상 시간 ≈ {total/3600:.1f}시간 (조기 종료 시 더 짧음)")
if RUN_E6:
    try:
        import segmentation_models_pytorch as smp
        t_pp = time_per_epoch(lambda: smp.UnetPlusPlus(**E6_KW, encoder_weights=None))
        print(f"U-Net++ 1 에폭 ≈ {t_pp:.1f}초 → E6 최대 ≈ {t_pp*EPOCHS_MAIN/3600:.1f}시간")
    except Exception as e:
        print("U-Net++ 측정 건너뜀:", repr(e))

## 8. 실험

### E1 — Baseline: U-Net + Cross Entropy, 증강 없음

| 설정 | 값 |
|---|---|
| 모델 | U-Net (base 16ch) |
| 손실 | CE |
| 증강 | 없음 |
| lr / batch | 1e-3 / 4 |

**가설**
- Pixel Accuracy는 높게(0.9 이상) 나오지만 mIoU는 상대적으로 낮을 것 — 그라운드·관중·광고판이 픽셀 대부분을 차지하기 때문.
- Ball, Goal Bar는 IoU가 0에 가까울 수 있음 (384×224에서 공은 지름 2~4픽셀).
- 증강이 없으므로 train loss는 계속 내려가지만 val loss는 일찍 바닥을 찍고 다시 오르는 **과적합 곡선**이 보일 것.

**관전 포인트**: best epoch가 언제인지, train/val loss 격차, 클래스별 IoU 중 0에 가까운 클래스.

In [ ]:
# ===== E1 =====
m_e1 = run_experiment("E1_UNet_CE", lambda: UNet(n_classes=NUM_CLASSES),
                      epochs=EPOCHS_MAIN, lr=1e-3, batch_size=4, loss_type="ce", augment=False, arch="U-Net")

### E2 — 손실 함수 변경: CE + Dice Loss

E1과 **손실 함수만** 다릅니다.

**가설**
- Dice가 클래스별 겹침을 동등한 비중으로 최적화하므로 **소수 클래스(Referee, Goalkeeper, Goal Bar, Ball) IoU가 상승**하고 mIoU가 E1보다 오를 것.
- Pixel Accuracy는 거의 변하지 않거나 오히려 소폭 하락할 수 있음 (작은 클래스를 더 과감히 예측하면서 FP 증가).

**관전 포인트**: mIoU 상승분이 어떤 클래스에서 왔는지 — 아래 E1 vs E2 클래스별 IoU 비교 표에서 확인.

In [ ]:
# ===== E2 =====
m_e2 = run_experiment("E2_UNet_CEDice", lambda: UNet(n_classes=NUM_CLASSES),
                      epochs=EPOCHS_MAIN, lr=1e-3, batch_size=4, loss_type="ce_dice", augment=False, arch="U-Net")
per_class_table().loc[["E1_UNet_CE", "E2_UNet_CEDice"]]

### E3 — 데이터 증강 적용 (Random Resized Crop + Horizontal Flip + Color Jitter)

E2와 **증강 여부만** 다릅니다.

**가설**
- train loss는 E2보다 **높게** 유지되지만(매번 다른 이미지를 보므로) val/test mIoU는 **상승** — 과적합 완화.
- best epoch가 E2보다 **늦어질 것** (더 오래 학습해도 일반화가 유지).
- Random Resized Crop의 확대 효과로 작은 객체(Ball, Referee) 인식이 좋아질 수 있음.
- 반면 Color Jitter가 유니폼 색 단서를 흐리면 Team A/B 구분은 정체되거나 떨어질 수도 있음.

**관전 포인트**: 학습 곡선에서 train/val 격차가 줄었는지, Team A/B·Goalkeeper IoU 변화.

In [ ]:
# ===== E3 =====
m_e3 = run_experiment("E3_UNet_CEDice_Aug", lambda: UNet(n_classes=NUM_CLASSES),
                      epochs=EPOCHS_MAIN, lr=1e-3, batch_size=4, loss_type="ce_dice", augment=True, arch="U-Net")
per_class_table().loc[["E2_UNet_CEDice", "E3_UNet_CEDice_Aug"]]

### E4 — 하이퍼파라미터 튜닝 (학습률 × 배치 크기)

E3 설정에서 **학습률 {1e-3, 3e-4} × 배치 크기 {4, 8}** 4가지 조합을 짧게(`EPOCHS_TUNE` 에폭) 학습해 **Validation mIoU로만** 비교합니다(테스트 세트는 보지 않음). 최적 조합으로 `EPOCHS_MAIN` 에폭 재학습한 것이 E4입니다. 에폭 수는 조기 종료와 best epoch 기록으로 함께 판단합니다.

**가설**
- 데이터가 70장이라 batch 8이면 에폭당 업데이트가 8번뿐 → 같은 에폭에서 **batch 4가 더 잘 수렴**할 것.
- 1e-3은 초반 수렴이 빠르고, 3e-4는 안정적이지만 짧은 학습에서는 덜 수렴했을 수 있음.
- 단, 튜닝이 짧은 학습 기준이므로 긴 학습에서 순위가 바뀔 수 있음.

**관전 포인트**: 조합 간 Val mIoU 차이가 얼마나 큰지(1%p 미만이면 사실상 차이 없음 = 기본값이 이미 적절).

In [ ]:
# ===== E4-1. 그리드 탐색 (Validation 기준) =====
TUNE_GRID = [(lr, bs) for lr in [1e-3, 3e-4] for bs in [4, 8]] if RUN_TUNING else []
tune_rows = []
for lr, bs in TUNE_GRID:
    seed_everything(SEED)
    name = f"tune_lr{lr:g}_bs{bs}"
    _, h, sec = train_model(UNet(n_classes=NUM_CLASSES), name, EPOCHS_TUNE, lr, bs, "ce_dice", True, verbose=False)
    tune_rows.append(dict(lr=lr, bs=bs, best_val_mIoU=h.val_miou.max(),
                          best_epoch=int(h.epoch[h.val_miou.idxmax()]), minutes=round(sec / 60, 1)))
    print(f"lr={lr:g}, bs={bs} → best val mIoU {h.val_miou.max():.4f} (ep {tune_rows[-1]['best_epoch']})")

if tune_rows:
    df_tune = pd.DataFrame(tune_rows).sort_values("best_val_mIoU", ascending=False).round(4)
    display(df_tune)
    BEST_LR, BEST_BS = float(df_tune.iloc[0].lr), int(df_tune.iloc[0].bs)
else:
    df_tune = pd.DataFrame(columns=["lr", "bs", "best_val_mIoU", "best_epoch", "minutes"])
    BEST_LR, BEST_BS = 1e-3, 4
    print("튜닝 생략 → 기본값 사용")
print(f"선택된 하이퍼파라미터: lr={BEST_LR:g}, batch={BEST_BS}")

In [ ]:
# ===== E4-2. 최적 조합으로 전체 학습 =====
m_e4 = run_experiment("E4_UNet_Tuned", lambda: UNet(n_classes=NUM_CLASSES),
                      epochs=EPOCHS_MAIN, lr=BEST_LR, batch_size=BEST_BS, loss_type="ce_dice", augment=True, arch="U-Net")

### E5 — 구조 개선: Attention U-Net

E4와 같은 하이퍼파라미터·손실·증강, **skip connection에 Attention Gate만 추가**합니다.

**가설**
- 게이트가 관중석·광고판의 복잡한 텍스처를 억제하고 선수·공 영역을 강조 → **Ball, Referee, Goalkeeper** 등 작은 객체 IoU 소폭 상승.
- 하지만 데이터가 적어 게이트 자체를 학습하기 어려울 수 있어 **전체 mIoU 차이는 ±1~2%p 수준**일 가능성. 파라미터 증가는 미미.

**관전 포인트**: mIoU 전체보다 **작은 객체 클래스의 IoU 변화**를 E4와 나란히 비교.

In [ ]:
# ===== E5 =====
if RUN_E5:
    m_e5 = run_experiment("E5_AttnUNet", lambda: UNet(n_classes=NUM_CLASSES, attention=True),
                          epochs=EPOCHS_MAIN, lr=BEST_LR, batch_size=BEST_BS, loss_type="ce_dice", augment=True,
                          arch="Attention U-Net")
    display(per_class_table().loc[["E4_UNet_Tuned", "E5_AttnUNet"]])
else:
    print("E5 건너뜀 (RUN_E5=False)")

### E6 — 구조 + 전이학습: U-Net++ (ResNet18 인코더, ImageNet 사전학습)

- **U-Net++** (Zhou et al., 2018): 인코더-디코더 사이를 **중첩된 dense skip 경로**로 연결해 인코더/디코더 특징 간 의미 차이(semantic gap)를 줄입니다.
- 인코더로 ImageNet 사전학습 **ResNet18**을 사용 (`segmentation_models_pytorch`). CPU 시간을 고려해 디코더 채널도 (128, 64, 32, 16, 16)으로 줄였습니다.
- 첫 실행 시 ImageNet 가중치(약 45MB)를 인터넷에서 내려받습니다.
- 사전학습 가중치를 크게 망가뜨리지 않도록 학습률을 **3e-4**로 낮추고, 배치·손실·증강은 E4와 동일하게 둡니다.

**가설**
- 100장 수준의 소량 데이터에선 **사전학습된 일반 시각 특징(경계·질감·사람 형태)** 의 효과가 커서 **6개 실험 중 최고 mIoU**를 낼 것.
- 특히 사람 계열 클래스(Referee, Coaches & Officials, Goalkeeper)의 개선 폭이 클 것.

**관전 포인트**: E5 대비 개선 폭이 "구조(E5 vs E4)"보다 훨씬 크다면 → 소량 데이터에서는 **구조보다 사전학습이 중요**하다는 결론.

In [ ]:
# ===== E6 (사전학습 가중치 다운로드 필요 — 실패 시 건너뜀) =====
if RUN_E6:
    try:
        import segmentation_models_pytorch as smp
        m_e6 = run_experiment("E6_UNetPP_R18",
                              lambda: smp.UnetPlusPlus(**E6_KW, encoder_weights="imagenet"),
                              epochs=EPOCHS_MAIN, lr=3e-4, batch_size=BEST_BS, loss_type="ce_dice", augment=True,
                              arch="U-Net++ (ResNet18, ImageNet)")
    except Exception as e:
        print("E6 건너뜀:", repr(e))
else:
    print("E6 건너뜀 (RUN_E6=False)")

## 9. 결과 비교 및 최종 모델

### 9-1. 전체 실험 요약
- **최종 모델은 Validation mIoU가 가장 높은 실험**으로 자동 선택합니다. 테스트 점수로 고르면 테스트 세트에 과적합한 선택이 되므로 피합니다.
- 클래스별 IoU 히트맵으로 각 개선 단계가 어느 클래스에 효과가 있었는지 한눈에 봅니다.

In [ ]:
# ===== 9-1. 요약표 & 클래스별 IoU 히트맵 =====
df_summary = summary_table()
display(df_summary)

df_cls = per_class_table("test")
fig, ax = plt.subplots(figsize=(15, 0.6 * len(df_cls) + 2))
im = ax.imshow(df_cls.values.astype(float), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(len(df_cls))); ax.set_yticklabels(df_cls.index)
for i in range(df_cls.shape[0]):
    for j in range(df_cls.shape[1]):
        v = df_cls.values[i, j]
        ax.text(j, i, "NaN" if np.isnan(v) else f"{v:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax); ax.set_title("Test IoU per class"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
df_summary[["val_mIoU", "test_mIoU", "test_mDice"]].plot.bar(ax=ax[0], rot=30, title="mIoU / mDice")
ax[0].set_ylim(0, 1)
df_summary[["test_CE", "test_DiceLoss"]].plot.bar(ax=ax[1], rot=30, title="Test Loss (CE / Dice)")
plt.tight_layout(); plt.show()

FINAL_NAME = df_summary.val_mIoU.idxmax()
print("최종 모델(Val mIoU 최고):", FINAL_NAME)

### 9-2. 최종 모델 테스트 평가 — 혼동행렬

행(정답)을 기준으로 정규화한 혼동행렬입니다. 대각선은 클래스별 Recall이며, **대각선 밖의 큰 값이 모델이 헷갈리는 클래스 쌍**입니다.

**관전 포인트**: Team A ↔ Goalkeeper A, Team A ↔ Team B, Referee ↔ Coaches & Officials, 작은 객체 → Ground로의 흡수.

In [ ]:
# ===== 9-2. 최종 모델 로드 & 혼동행렬 =====
def build_model(name):
    arch = RESULTS[name]["arch"]
    if arch.startswith("U-Net++"):
        import segmentation_models_pytorch as smp
        return smp.UnetPlusPlus(**E6_KW, encoder_weights=None)
    return UNet(n_classes=NUM_CLASSES, attention=(arch == "Attention U-Net"))

final_model = build_model(FINAL_NAME).to(DEVICE)
final_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, f"{FINAL_NAME}.pt"), map_location=DEVICE))
final_test = evaluate(final_model, test_dl)
print(f"[{FINAL_NAME}] Test — PixAcc {final_test['pixel_acc']:.4f} | mIoU {final_test['miou']:.4f} | "
      f"mDice {final_test['mdice']:.4f} | CE {final_test['ce_loss']:.4f} | DiceLoss {final_test['dice_loss']:.4f}")
display(pd.DataFrame({"IoU": final_test["iou"], "Dice": final_test["dice"]}, index=CLASS_NAMES).round(4))

cm = final_test["cm"].astype(np.float64)
cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm_norm[i, j] >= 0.01:
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground Truth"); ax.set_title(f"{FINAL_NAME} — Normalized Confusion Matrix")
plt.colorbar(im); plt.tight_layout(); plt.show()

# 가장 많이 혼동되는 쌍 Top 8 (대각선 제외)
off = [(CLASS_NAMES[i], CLASS_NAMES[j], cm_norm[i, j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j]
pd.DataFrame(sorted(off, key=lambda t: -t[2])[:8], columns=["정답", "예측", "비율"]).round(3)

### 9-3. 예측 시각화
테스트 이미지마다 **원본 / 정답 / 예측 / 오류 맵**(빨강 = 틀린 픽셀)을 나란히 보여줍니다. 이미지별 mIoU도 함께 표시해 잘된 장면과 실패한 장면을 비교합니다.

In [ ]:
# ===== 9-3. 테스트 예측 시각화 =====
@torch.no_grad()
def predict(model, img_uint8):
    """img_uint8: (H,W,3) RGB, 이미 IMG_H×IMG_W로 리사이즈된 이미지 → (H,W) 클래스 맵"""
    model.eval()
    x = TF.normalize(torch.from_numpy(img_uint8).permute(2, 0, 1).float() / 255.0, MEAN, STD)[None].to(DEVICE)
    with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
        return model(x).argmax(1)[0].cpu().numpy().astype(np.uint8)

def image_miou(gt, pr):
    C = NUM_CLASSES; ok = gt != IGNORE_INDEX
    cm = np.bincount(gt[ok].astype(int) * C + pr[ok], minlength=C * C).reshape(C, C)
    return metrics_from_cm(cm)["miou"]

preds = [predict(final_model, x) for x in X_te]
scores = [image_miou(g, p) for g, p in zip(Y_te, preds)]
order = np.argsort(scores)
show = list(order[-3:][::-1]) + list(order[:2])           # 잘된 3장 + 못한 2장

fig, axes = plt.subplots(len(show), 4, figsize=(22, 3.3 * len(show)))
for r, i in enumerate(show):
    err = np.zeros((*Y_te[i].shape, 3), np.uint8)
    err[(preds[i] != Y_te[i]) & (Y_te[i] != IGNORE_INDEX)] = (255, 0, 0)
    for c, (im, t) in enumerate([(X_te[i], f"{NAMES[idx_te[i]]}"), (colorize(Y_te[i]), "Ground Truth"),
                                 (colorize(preds[i]), f"Prediction (mIoU {scores[i]:.3f})"), (err, "Error (red)")]):
        axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=10); axes[r, c].axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(4, 4)); show_legend(ax); plt.show()

### 9-4. 새 이미지 추론 함수 & 최종 모델 저장
- `segment_image(path)`: 원본 크기 이미지를 받아 리사이즈 → 예측 → 원본 크기로 복원(최근접 보간)한 컬러 마스크와 오버레이를 반환합니다.
- 최종 가중치(`final_model.pt`)와 재사용에 필요한 메타데이터(클래스명·팔레트·입력 크기·정규화 값), 결과표 CSV를 `outputs/`에 저장합니다.

In [ ]:
# ===== 9-4. 추론 함수 & 저장 =====
def segment_image(path, model=final_model, alpha=0.5):
    rgb = read_rgb(path); H0, W0 = rgb.shape[:2]
    pred = predict(model, cv2.resize(rgb, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA))
    pred = cv2.resize(pred, (W0, H0), interpolation=cv2.INTER_NEAREST)
    color = colorize(pred)
    return pred, color, (rgb * (1 - alpha) + color * alpha).astype(np.uint8)

_, color, overlay = segment_image(pairs[idx_te[0]][0])
plt.figure(figsize=(14, 5)); plt.imshow(overlay); plt.axis("off"); plt.title("Inference overlay (original resolution)"); plt.show()

torch.save(final_model.state_dict(), os.path.join(SAVE_DIR, "final_model.pt"))
json.dump({"final_experiment": FINAL_NAME, "arch": RESULTS[FINAL_NAME]["arch"], "class_names": CLASS_NAMES,
           "palette": PALETTE.tolist(), "img_size_hw": [IMG_H, IMG_W], "mean": MEAN, "std": STD,
           "ignore_index": IGNORE_INDEX},
          open(os.path.join(SAVE_DIR, "final_model_meta.json"), "w"), ensure_ascii=False, indent=2)
df_summary.to_csv(os.path.join(SAVE_DIR, "results_summary.csv"))
per_class_table("test").to_csv(os.path.join(SAVE_DIR, "results_per_class_iou.csv"))
df_tune.to_csv(os.path.join(SAVE_DIR, "results_tuning.csv"), index=False)
print("저장 완료:", sorted(os.listdir(SAVE_DIR)))

## 10. 결과 공유용 요약 출력

아래 셀의 출력 전체를 복사해 공유해 주시면, 가설 대비 실제 결과를 바탕으로 **결과 해설**(가설 검증, 클래스별 분석, 개선 방향)을 작성해 드리겠습니다. 학습 곡선·혼동행렬·예측 시각화 이미지 몇 장을 함께 첨부해 주시면 더 정확한 해설이 가능합니다.

In [ ]:
# ===== 10. 결과 요약 (복사해서 공유) =====
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("=" * 30, "환경", "=" * 30)
print(f"device={DEVICE} ({torch.cuda.get_device_name(0) if USE_AMP else platform.processor()}) | threads={torch.get_num_threads()} | img={IMG_W}x{IMG_H} | base_ch={BASE_CH} | "
      f"split train/val/test={len(idx_tr)}/{len(idx_va)}/{len(idx_te)} | QUICK_MODE={QUICK_MODE}")
print("\n" + "=" * 30, "클래스 분포", "=" * 30); print(df_dist.to_string(index=False))
print("\n" + "=" * 30, "테스트 세트 클래스 등장 수", "=" * 30); print(presence["test"].to_string())
print("\n" + "=" * 30, "실험 요약", "=" * 30); print(df_summary.to_string())
print("\n" + "=" * 30, "하이퍼파라미터 탐색 (Val)", "=" * 30); print(df_tune.to_string(index=False))
print("\n" + "=" * 30, "클래스별 Test IoU", "=" * 30); print(per_class_table("test").to_string())
print("\n" + "=" * 30, f"최종 모델: {FINAL_NAME}", "=" * 30)
print(f"Test PixAcc {final_test['pixel_acc']:.4f} | mIoU {final_test['miou']:.4f} | mDice {final_test['mdice']:.4f} "
      f"| CE {final_test['ce_loss']:.4f} | DiceLoss {final_test['dice_loss']:.4f}")
print("혼동 Top:", "; ".join(f"{a}→{b} {v:.2f}" for a, b, v in sorted(off, key=lambda t: -t[2])[:6]))

## 11. 결과 해설

> 실험 실행 후 결과를 공유해 주시면 이 섹션에 작성합니다.
> - 가설별 검증 결과 (E1~E6)
> - 클래스별 IoU·혼동 패턴 분석
> - 한계와 추가 개선 방향